# Group Project - Safe routes home at night 

**DS105A Group Project – Data for Data Science (Autumn Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Group Notebook**
- 👤 Team Name: CtrlAltElite
- 🎯 Purpose: NB03 will be used to create all of our data visualisations that will display safe routes home. 
</div>

## Data Visualisation Pipeline

1. Understand the data 
   - Do some quick checks of the database, so I can see what I am working with
2. Create a scoring system
   - Create an arbitrary scoring system that looks at number of crimes, distance and time
   - Apply weightings to each crime, a more severe crime has a higher weighting, thus more dangerous 
   - Create an overall scoring system that considers all metrics 
   - Output a score for each route 
3. Test the usability of the scoring system
   - Create tables that show all the route data as well as the scoring metrics 
4. Create plots and maps
   - ss

In [39]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

## Step 1 - Understand the Database


In [40]:
DB_PATH = Path("data/processed/nightclub_safety.db")
conn = sqlite3.connect(str(DB_PATH))

# List all tables
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

print("Tables in database:")
display(tables)

# Inspect each table 
for table in tables["name"]:
    print(f"TABLE: {table}")

    # schema
    schema = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
    print("\nColumns & types:")
    display(schema[["name", "type"]])

    # row count
    count = pd.read_sql_query(f"SELECT COUNT(*) AS rows FROM {table};", conn)
    print("\nNumber of rows:")
    display(count)

    # preview data
    preview = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 5;", conn)
    print("\nExample rows:")
    display(preview)

conn.close()


Tables in database:


,name
0,areas
1,sqlite_sequence
2,clubs
3,destinations
4,routes
5,crimes
6,route_crimes


TABLE: areas

Columns & types:


,name,type
0,area_id,INTEGER
1,name,TEXT



Number of rows:


,rows
0,3


... (34 outputs left out)


Number of rows:


,rows
0,4317



Example rows:


,route_id,crime_id
0,4,1
1,4,2
2,4,3
3,4,4
4,4,5


In [41]:
conn = sqlite3.connect("data/processed/nightclub_safety.db")

df = pd.read_sql_query(
    "SELECT DISTINCT category FROM crimes",
    conn
)

conn.close()
display(df)

,category
0,violent-crime
1,public-order
2,anti-social-behaviour


### What have I found and what I intend to use 

- The different types of crimes committed, this is very useful as I can place a weighting on the different types of crimes, thus more serious crimes will further decrease the safety score than if it was a less serious crime. 
- I also will use simple measures such as distance and time, but these will need to be specified by number of crimes along the routes. This will prevent a 100m walk with 1 crime being viewed as safer than a 1km walk with 1 crime.  

## Step 2 - Create a Scoring System 


In [42]:
DB_PATH = Path("data/processed/nightclub_safety.db")
assert DB_PATH.exists(), f"DB not found: {DB_PATH.resolve()}"

conn = sqlite3.connect(str(DB_PATH))

# Extract data from database 
query = """
SELECT
    a.name AS area,
    c.club_id,
    c.displayName AS club_name,
    d.destination_id,
    d.displayName AS destination_name,
    r.route_id,
    r.distanceMeters AS distance_m,
    r.durationSeconds AS duration_s,
    COUNT(rc.crime_id) AS crime_count
FROM areas a
JOIN clubs c ON a.area_id = c.area_id
JOIN destinations d ON c.club_id = d.club_id
JOIN routes r ON d.destination_id = r.destination_id
LEFT JOIN route_crimes rc ON r.route_id = rc.route_id
GROUP BY
    a.name,
    c.club_id, c.displayName,
    d.destination_id, d.displayName,
    r.route_id, r.distanceMeters, r.durationSeconds;
"""

df_scored = pd.read_sql_query(query, conn)
conn.close()

# Create new metrics and clean time metric 
df_scored["distance_m"] = pd.to_numeric(df_scored["distance_m"], errors="coerce")

# Duration not given as interger example '16s'
df_scored["duration_s"] = (
    df_scored["duration_s"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)
)
df_scored["duration_s"] = pd.to_numeric(df_scored["duration_s"], errors="coerce")

df_scored["crime_count"] = pd.to_numeric(df_scored["crime_count"], errors="coerce").fillna(0)

# derived units
df_scored["distance_km"] = df_scored["distance_m"] / 1000.0
df_scored["duration_min"] = df_scored["duration_s"] / 60.0

# rates 
df_scored["crimes_per_km"] = np.where(
    df_scored["distance_km"] > 0,
    df_scored["crime_count"] / df_scored["distance_km"],
    np.nan
)
df_scored["crimes_per_min"] = np.where(
    df_scored["duration_min"] > 0,
    df_scored["crime_count"] / df_scored["duration_min"],
    np.nan
)

df_scored.replace([np.inf, -np.inf], np.nan, inplace=True)
df_scored["crimes_per_km"] = df_scored["crimes_per_km"].fillna(0)
df_scored["crimes_per_min"] = df_scored["crimes_per_min"].fillna(0)

#Normalise each area 
def minmax_series(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)

df_scored["crime_km_norm"]  = df_scored.groupby("area")["crimes_per_km"].transform(minmax_series)
df_scored["crime_min_norm"] = df_scored.groupby("area")["crimes_per_min"].transform(minmax_series)
df_scored["dist_norm"]      = df_scored.groupby("area")["distance_km"].transform(minmax_series)

# Duration might be missing for some routes
df_scored["dur_norm"]       = df_scored.groupby("area")["duration_min"].transform(minmax_series)

#Calculate a safety score with a weighting system 
w_crime_km, w_crime_min, w_dist, w_dur = 0.3, 0.3, 0.2, 0.2

df_scored["risk_score"] = (
    w_crime_km  * df_scored["crime_km_norm"] +
    w_crime_min * df_scored["crime_min_norm"] +
    w_dist      * df_scored["dist_norm"] +
    w_dur       * df_scored["dur_norm"]
)

df_scored["safety_score"] = 1 - df_scored["risk_score"]
df_scored["safety_score_100"] = (df_scored["safety_score"] * 100).round(2)

# Ranks within each area
df_scored["rank_in_area"] = (
    df_scored.sort_values(["area", "safety_score_100", "crime_count"], ascending=[True, False, True])
            .groupby("area")
            .cumcount() + 1
)




In [43]:
AREA = "Dalston"   



dalston = (
    df_scored[df_scored["area"] == AREA]
    .sort_values(["safety_score_100", "crime_count"], ascending=[False, True])
    .loc[:, [
        "rank_in_area",
        "club_name",
        "destination_name",
        "route_id",
        "distance_km",
        "duration_min",
        "crime_count",
        "crimes_per_km",
        "crimes_per_min",
        "safety_score_100",
    ]]
   
    .copy()
)


dalston["distance_km"] = dalston["distance_km"].round(3)
dalston["duration_min"] = dalston["duration_min"].round(1)
dalston["crimes_per_km"] = dalston["crimes_per_km"].round(2)
dalston["crimes_per_min"] = dalston["crimes_per_min"].round(3)

display(dalston.reset_index(drop=True))

,rank_in_area,club_name,destination_name,route_id,distance_km,duration_min,crime_count,crimes_per_km,crimes_per_min,safety_score_100
0,1,Dalston Superstore,Rio Cinema (Stop T),12,0.013,0.4,0,0.00,0.000,99.86
1,2,Farr's Dalston,Dalston Lane /Dalston Junction Station (Stop K),1,0.023,0.3,0,0.00,0.000,99.81
2,3,Club No.8,Shacklewell Lane / Kingsland High Street (Stop W),47,0.039,0.5,0,0.00,0.000,99.12
3,4,Farr's Dalston,Dalston Junction Station (Stop G),3,0.043,0.8,0,0.00,0.000,98.72
4,5,The Speakeasy,Dalston Junction Station (Stop G),66,0.046,0.8,0,0.00,0.000,98.59
...,...,...,...,...,...,...,...,...,...,...
72,73,Dalston Superstore,Stoke Newington Police Station (Stop R),21,0.871,12.5,128,146.96,10.254,49.97
73,74,Farr's Dalston,Dalston Kingsland Station (Stop P),8,0.359,5.1,123,342.62,24.039,48.57
74,75,The Karaoke Hole,Dalston Junction Station (Stop G),58,0.420,5.8,144,342.86,24.828,45.82
75,76,The Speakeasy,Dalston Kingsland Station (Stop P),75,0.281,4.0,128,455.52,31.605,39.36


💭 **Personal Reflection:** 
- database waffle link to lectures 

In [44]:
DB_PATH = Path("data/processed/nightclub_safety.db")
assert DB_PATH.exists(), f"DB not found: {DB_PATH.resolve()}"

# Severity weights 
CRIME_WEIGHTS = {
    # Higher severity
    "violent-crime": 3.0,
    "public-order": 2.0,
    "anti-social-behaviour": 1.0,
    
}


w_crime_km, w_crime_min, w_dist, w_dur = 0.3, 0.3, 0.2, 0.2


df_raw["crime_weight"] = (
    df_raw["category"]
    .map(CRIME_WEIGHTS)
    .fillna(DEFAULT_CRIME_WEIGHT)
)

df_raw.loc[df_raw["category"].isna(), "crime_weight"] = 0.0


df_scored_weighted = (
    df_raw
    .groupby(
        ["area", "club_id", "club_name", "destination_id", "destination_name",
         "route_id", "distance_m", "duration_s"],
        as_index=False
    )
    .agg(
        weighted_crime_score=("crime_weight", "sum"),
        crime_count=("category", lambda s: s.notna().sum())
    )
)


df_scored_weighted["distance_km"] = df_scored_weighted["distance_m"] / 1000.0
df_scored_weighted["duration_min"] = df_scored_weighted["duration_s"] / 60.0

df_scored_weighted["weighted_crimes_per_km"] = np.where(
    df_scored_weighted["distance_km"] > 0,
    df_scored_weighted["weighted_crime_score"] / df_scored_weighted["distance_km"],
    np.nan
)

df_scored_weighted["weighted_crimes_per_min"] = np.where(
    df_scored_weighted["duration_min"] > 0,
    df_scored_weighted["weighted_crime_score"] / df_scored_weighted["duration_min"],
    np.nan
)

df_scored_weighted.replace([np.inf, -np.inf], np.nan, inplace=True)
df_scored_weighted["weighted_crimes_per_km"] = df_scored_weighted["weighted_crimes_per_km"].fillna(0)
df_scored_weighted["weighted_crimes_per_min"] = df_scored_weighted["weighted_crimes_per_min"].fillna(0)


def minmax_series(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)

df_scored_weighted["crime_km_norm"]  = df_scored_weighted.groupby("area")["weighted_crimes_per_km"].transform(minmax_series)
df_scored_weighted["crime_min_norm"] = df_scored_weighted.groupby("area")["weighted_crimes_per_min"].transform(minmax_series)
df_scored_weighted["dist_norm"]      = df_scored_weighted.groupby("area")["distance_km"].transform(minmax_series)
df_scored_weighted["dur_norm"]       = df_scored_weighted.groupby("area")["duration_min"].transform(minmax_series)


df_scored_weighted["risk_score"] = (
    w_crime_km  * df_scored_weighted["crime_km_norm"] +
    w_crime_min * df_scored_weighted["crime_min_norm"] +
    w_dist      * df_scored_weighted["dist_norm"] +
    w_dur       * df_scored_weighted["dur_norm"]
)

df_scored_weighted["safety_score"] = 1 - df_scored_weighted["risk_score"]
df_scored_weighted["safety_score_100"] = (df_scored_weighted["safety_score"] * 100).round(2)

# Rank within each area (optional but useful)
df_scored_weighted["rank_in_area"] = (
    df_scored_weighted.sort_values(["area", "safety_score_100", "crime_count"], ascending=[True, False, True])
                      .groupby("area")
                      .cumcount() + 1
)




In [47]:
# Display safest routes with weighted scores 
areas = ["Dalston", "Hackney_Wick", "Bermondsey"]

cols = [
    "rank_in_area",
    "club_name",
    "destination_name",
    "route_id",
    "distance_km",
    "duration_min",
    "crime_count",
    "weighted_crime_score",
    "weighted_crimes_per_km",
    "weighted_crimes_per_min",
    "safety_score_100",
]

tables_by_area = {}

for area in areas:
    t = (
        df_scored_weighted[df_scored_weighted["area"] == area]
        .sort_values("rank_in_area")
        .loc[:, cols]
        .copy()
    )

    # formatting
    t["distance_km"] = t["distance_km"].round(3)
    t["duration_min"] = t["duration_min"].round(1)
    t["weighted_crimes_per_km"] = t["weighted_crimes_per_km"].round(2)
    t["weighted_crimes_per_min"] = t["weighted_crimes_per_min"].round(3)

    tables_by_area[area] = t.reset_index(drop=True)

    print(f"\n===== {area}: Ranked routes =====")
    display(tables_by_area[area])

# Now you also have:
# tables_by_area["Dalston"], tables_by_area["Hackney"], tables_by_area["Bermondsey"]





===== Dalston: Ranked routes =====


,rank_in_area,club_name,destination_name,route_id,distance_km,duration_min,crime_count,weighted_crime_score,weighted_crimes_per_km,weighted_crimes_per_min,safety_score_100
0,1,Dalston Superstore,Rio Cinema (Stop T),12,0.013,0.4,0,0.0,0.00,0.000,99.86
1,2,Farr's Dalston,Dalston Lane /Dalston Junction Station (Stop K),1,0.023,0.3,0,0.0,0.00,0.000,99.81
2,3,Club No.8,Shacklewell Lane / Kingsland High Street (Stop W),47,0.039,0.5,0,0.0,0.00,0.000,99.12
3,4,Farr's Dalston,Dalston Junction Station (Stop G),3,0.043,0.8,0,0.0,0.00,0.000,98.72
4,5,The Speakeasy,Dalston Junction Station (Stop G),66,0.046,0.8,0,0.0,0.00,0.000,98.59
...,...,...,...,...,...,...,...,...,...,...,...
72,73,Club No.8,Stoke Newington Police Station (Stop R),56,0.816,11.4,135,313.0,383.58,27.336,40.78
73,74,Dalston Superstore,Stoke Newington Police Station (Stop R),21,0.871,12.5,128,314.0,360.51,25.154,40.21
74,75,The Karaoke Hole,Dalston Junction Station (Stop G),58,0.420,5.8,144,257.0,611.90,44.310,39.44
75,76,The Speakeasy,Dalston Kingsland Station (Stop P),75,0.281,4.0,128,212.0,754.45,52.346,35.26



===== Hackney_Wick: Ranked routes =====


,rank_in_area,club_name,destination_name,route_id,distance_km,duration_min,crime_count,weighted_crime_score,weighted_crimes_per_km,weighted_crimes_per_min,safety_score_100
0,1,MOTH Club,Hackney Town Hall (Stop B),79,0.135,1.9,0,0.0,0.00,0.000,96.94
1,2,MOTH Club,Hackney Central / Mare Street (Stop G),81,0.258,4.1,0,0.0,0.00,0.000,92.33
2,3,MOTH Club,Hackney Central,82,0.342,5.2,0,0.0,0.00,0.000,89.69
3,4,The Glove That Fits,Trelawney Estate (Stop C),84,0.484,6.9,6,6.0,12.40,0.874,84.45
4,5,The Glove That Fits,Hackney Central Station (Stop H),87,0.708,9.7,8,11.0,15.54,1.136,77.22
5,6,Grow,St Mary Of Eton Church (Stop N),90,0.734,10.2,6,6.0,8.17,0.587,76.71
6,7,MOTH Club,Richmond Road /Martello Street (Stop LS),80,0.346,5.0,44,68.0,196.53,13.645,75.74
7,8,Fabwick,St Mary Of Eton Church (Stop N),83,0.783,10.9,10,16.0,20.43,1.468,74.21
8,9,The Glove That Fits,Brooksby's Walk / Homerton Station (Stop HB),85,0.768,11.1,14,34.0,44.27,3.054,72.51
9,10,The Glove That Fits,Mare Street / Narrow Way (Stop R),86,0.765,10.8,69,117.0,152.94,10.800,65.03



===== Bermondsey: Ranked routes =====


,rank_in_area,club_name,destination_name,route_id,distance_km,duration_min,crime_count,weighted_crime_score,weighted_crimes_per_km,weighted_crimes_per_min,safety_score_100
0,1,London Bridge Rooftop Bar,London Bridge,125,0.095,1.4,0,0.0,0.00,0.000,100.00
1,2,The Barrel Project,Tanner Street (Stop B),101,0.132,1.8,0,0.0,0.00,0.000,98.98
2,3,Electric Shuffle London Bridge,London Bridge,144,0.152,2.0,0,0.0,0.00,0.000,98.48
3,4,Electric Shuffle London Bridge,Hays Galleria (Stop H),146,0.152,2.3,0,0.0,0.00,0.000,98.22
4,5,N1 Bar - London Bridge,London Bridge Bus Station (Stop A),92,0.170,2.7,0,0.0,0.00,0.000,97.53
...,...,...,...,...,...,...,...,...,...,...,...
70,71,London Bridge Rooftop Bar,Borough,133,0.753,11.0,59,152.0,201.86,13.756,37.35
71,72,Bermondsey Bierkeller,London Bridge bus station,117,0.238,4.2,29,74.0,310.92,17.689,34.94
72,73,Electric Shuffle London Bridge,Borough,154,1.154,16.4,71,183.0,158.58,11.170,34.42
73,74,Amazing Grace London Bridge,Borough,141,0.622,9.1,59,152.0,244.37,16.642,32.29
